<a href="https://colab.research.google.com/github/karye/Liu-labb1/blob/main/Lab_1_Tic_Tac_Toe_simple.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 1 - Tre i rad

I den här laborationen ska du utforska kunskapsbaserade AI-agenter för tvåspelarspel. Mer specifikt ska du se hur AI-agenter kan använda logik och planering för att spela Tre i rad.

Laborationen börjar med att presentera spelet Tre i rad (som Python-kod) och går sedan igenom ett antal olika typer av AI-agenter:
* Slumpmässig agent
* Regelbaserad agent
* Planeringsagent (sökning)
* Motståndarspelarens planeringsagent (Minimax)
och du bjuds in att implementera din egen regelbaserade agent.
Som bonusuppgift utmanas du (inte obligatoriskt!) att implementera en stokastisk planeringsagent med Monte Carlo Tree Search (MCTS).

**Mål:** Undersök begränsningar, styrkor och svagheter hos olika AI-agentmetoder i Tre i rad.

**Gör labarna uppifrån och ner.**
<br />
**Du behöver inte läsa längre fram än det avsnitt du jobbar med just nu.**

<br />
<hr />

### **Författare**
Mathias Ahlgren <br />
Mattias Tiger, mattias.tiger@liu.se

### **Licens**
CC BY-NC-SA 4.0 <br />
https://creativecommons.org/licenses/by-nc-sa/4.0/

Laborationen (Notebook: kod och instruktioner) är fri att använda, dela, ändra och skapa egna versioner av för dina egna elever. Den enda begränsningen för denna version och alla nya versioner är att 1) samma licens (CC BY-NC-SA 4.0) används, 2) att ovanstående författare anges som upphovsmän till originalversionen och 3) att den inte får användas i sammanhang där en person eller ett företag måste betala för att få tillgång till eller använda versionen.
<hr />

# Tre i rad - Lab 1.0 - Spelet
Vi börjar med att sätta upp spelplanen, skapa det grafiska gränssnittet och koppla ihop händelselyssnare för knapparna.

### Steg 1: Inställningar och initialisering

* **Uppgift 1.0.1:** Läs igenom koden snabbt. Fokusera på funktionsnamnen och deras beskrivningar.

  * Fråga 1: Vad är skillnaden mellan *check_winner(...)* och *check_tile_status(...)*?
    * check_tile_status() kollar om det går att göra ett drag.
  * Fråga 2: Vad bestämmer storleken på spelplanen?
    * Konstanten BOARD_SIZE anger storleken, dvs 3x3

* **Uppgift 1.0.2:** Kör kodcellen för att starta spelet.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import random

# Inställning av spelplanen
BOARD_SIZE = 3
board = [[0] * BOARD_SIZE for _ in range(BOARD_SIZE)]

def check_winner(board):
    '''
    Kontrollerar om det finns en vinnare på spelplanen.

    Argument:
        board: Aktuell spelplan (2D-lista).

    Returnerar:
      1 för spelare 1
      2 för spelare 2
      3 för oavgjort
      None om spelet pågår.
    '''

    # Kolla rader och kolumner efter en vinnare
    for i in range(BOARD_SIZE):
        if all(board[i][j] == board[i][0] for j in range(BOARD_SIZE)) and board[i][0] != 0:
            return board[i][0]
        if all(board[j][i] == board[0][i] for j in range(BOARD_SIZE)) and board[0][i] != 0:
            return board[0][i]

    # Kolla diagonalerna efter en vinnare
    if all(board[k][k] == board[0][0] for k in range(BOARD_SIZE)) and board[0][0] != 0:
        return board[0][0]
    if all(board[k][BOARD_SIZE-1-k] == board[0][BOARD_SIZE-1] for k in range(BOARD_SIZE)) and board[0][BOARD_SIZE-1] != 0:
        return board[0][BOARD_SIZE-1]

    # Kolla om det är oavgjort (ingen vinnare och inga tomma rutor)
    if all(board[i][j] != 0 for i in range(BOARD_SIZE) for j in range(BOARD_SIZE)):
        return 3

    return None

def check_tile_status(board, row, col, player):
    '''
    Simulerar ett drag på den angivna positionen (row, col) och kontrollerar om det leder till vinst eller oavgjort.

    Argument:
        board: Aktuell spelplan (2D-lista).
        row: Radindex för rutan att kontrollera.
        col: Kolumnindex för rutan att kontrollera.
        current_player: Spelaren som gör draget (1 eller 2).

    Returnerar:
        1 om Spelare 1 vinner,
        2 om Spelare 2 vinner,
        3 för oavgjort,
        None om spelet pågår eller om rutan är upptagen.
    '''

    if board[row][col] != 0:
        return None  # Rutan är redan fylld, inget resultat.

    # Lägg tillfälligt draget för current_player
    board[row][col] = player

    # Kolla raden efter en vinnare
    if all(board[row][j] == player for j in range(BOARD_SIZE)):
        board[row][col] = 0  # Ångra draget
        return player

    # Kolla kolumnen efter en vinnare
    if all(board[j][col] == player for j in range(BOARD_SIZE)):
        board[row][col] = 0  # Ångra draget
        return player

    # Kolla huvuddiagonalen om tillämpligt
    if row == col and all(board[k][k] == player for k in range(BOARD_SIZE)):
        board[row][col] = 0  # Ångra draget
        return player

    # Kolla antidiagonalen om tillämpligt
    if row + col == BOARD_SIZE - 1 and all(board[k][BOARD_SIZE-1-k] == player for k in range(BOARD_SIZE)):
        board[row][col] = 0  # Ångra draget
        return player

    # Kolla om det är oavgjort (ingen vinnare och inga tomma rutor)
    if all(board[i][j] != 0 for i in range(BOARD_SIZE) for j in range(BOARD_SIZE)):
        board[row][col] = 0  # Ångra draget
        return 3  # Oavgjort

    board[row][col] = 0  # Ångra draget om ingen vinst/oavgjort
    return None

players = [
        ('Människa', 'human')
    ]

# Rullgardinsmeny för spelare 1
player1_dropdown = widgets.Dropdown(
    options=players,
    value='human',
    description='Spelare1 (X)',
)

# Rullgardinsmeny för spelare 2
player2_dropdown = widgets.Dropdown(
    options=players,
    value='human',
    description='Spelare2 (O)',
)

# Initiera den globala statusetiketten här
ai_status_label = widgets.Label(value="")

def create_buttons():
    ''' Skapar och returnerar en lista med knappar '''
    return [[widgets.Button(description='', layout=widgets.Layout(width='80px', height='80px')) for _ in range(BOARD_SIZE)] for _ in range(BOARD_SIZE)]

buttons = create_buttons()

def on_button_click(i, j):
    ''' Händelse för att göra ett drag som människa och spela AI:ns nästa drag '''
    global current_player, board

    if board[i][j] == 0:  # Om rutan är tom
        board[i][j] = current_player
        buttons[i][j].description = 'X' if current_player == 1 else 'O'
        buttons[i][j].disabled = True
        buttons[i][j].style.button_color = 'black'

        winner = check_winner(board)
        if winner:
            display_result(winner)
            return

        # Byt tur
        current_player = 1 if current_player == 2 else 2

        # Om den aktuella spelaren är en människa stannar vi och väntar på ett klick
        if get_current_player_type() == 'human':
            return

        # Stanna och vänta på "steg"-knappen
        if player1_dropdown.value != 'human' and player2_dropdown.value != 'human':
            return

        # Om det är AI:ns tur, gör draget direkt
        if get_current_player_type() != 'human':
            ai_move = get_ai_move(get_current_player_dropdown())
            if ai_move:
                on_button_click(ai_move[0], ai_move[1])

def get_ai_move(player_dropdown):
    '''
    Hämtar AI:ns nästa drag baserat på vald strategi.
    '''
    global ai_status_label, current_player, board

    selected_strategy = player_dropdown.value
    ai_status_label.value = "AI tänker..."

    ai_move = None

    if selected_strategy == 'random':
        ai_move = find_best_move_random(board)
    elif selected_strategy == 'my_ai_strategy':
        ai_move = find_best_move(board, current_player)
    elif selected_strategy == 'rule_based':
        ai_move = find_best_move_rule_based(board, current_player)
    elif selected_strategy == 'minimax':
        ai_move = find_best_move_min_max(board, current_player, use_alpha_beta=True)
    elif selected_strategy == 'dfs':
        ai_move = find_best_move_dfs(board, current_player)
    elif selected_strategy == 'bfs':
        ai_move = find_best_move_bfs(board, current_player)

    ai_status_label.value = "AI är klar!"
    return ai_move

def get_current_player_type():
    '''
    Avgör om den aktuella spelaren är AI eller människa.
    '''
    return player1_dropdown.value if current_player == 1 else player2_dropdown.value

def get_current_player_dropdown():
    '''
    Hämtar rätt rullgardinsmeny för den aktuella spelaren (Spelare 1 eller Spelare 2).
    '''
    return player1_dropdown if current_player == 1 else player2_dropdown

def display_result(winner):
    '''
    Visar resultatet av spelet.
    '''
    result = ""
    if winner == 1:
        result = "Spelare 1 vinner" if player1_dropdown.value == 'human' else "AI (Spelare 1) vinner"
    elif winner == 2:
        result = "Spelare 2 vinner" if player2_dropdown.value == 'human' else "AI (Spelare 2) vinner"
    elif winner == 3:
        result = "Det är oavgjort"

    result_label.value = result
    disable_all_buttons()

def disable_all_buttons():
    '''
    Inaktiverar alla knappar.
    '''
    for i in range(BOARD_SIZE):
        for j in range(BOARD_SIZE):
            buttons[i][j].disabled = True

def step_game(_=None):
    global current_player
    '''
    Funktion för att stega igenom AI mot AI-drag när "Steg"-knappen trycks.
    '''
    if player1_dropdown.value != 'human' and player2_dropdown.value != 'human': # Om AI mot AI inaktiverar vi knapparna
        disable_all_buttons()

    if get_current_player_type() != 'human':
        ai_move = get_ai_move(get_current_player_dropdown())
        if ai_move:
            on_button_click(ai_move[0], ai_move[1])

def new_game(_=None):
    '''
    Startar och ritar upp ett nytt spel.
    '''
    global board, current_player, buttons, result_label, ai_status_label
    clear_output()

    board = [[0] * BOARD_SIZE for _ in range(BOARD_SIZE)]
    current_player = 1

    buttons = create_buttons()

    for i in range(BOARD_SIZE):
        for j in range(BOARD_SIZE):
            buttons[i][j].on_click(lambda btn, i=i, j=j: on_button_click(i, j))

    grid = widgets.GridBox(children=[button for row in buttons for button in row],
                          layout=widgets.Layout(grid_template_columns=f"repeat({BOARD_SIZE}, 90px)"))

    result_label = widgets.Label(value="")
    ai_status_label.value = ""

    new_game_button = widgets.Button(description="Nytt spel", layout=widgets.Layout(width='200px', height='40px'))
    step_button = widgets.Button(description="Steg", layout=widgets.Layout(width='200px', height='40px'))

    new_game_button.on_click(new_game)
    step_button.on_click(step_game)

    buttons_box = widgets.HBox([new_game_button, step_button], layout=widgets.Layout(justify_content='flex-start'))

    display(player1_dropdown)
    display(player2_dropdown)
    display(grid)
    display(result_label)
    display(ai_status_label)
    display(buttons_box)

### Steg 2: Skapa det grafiska gränssnittet

(Kör kodcellen)

* **Uppgift 1.0.3:** Spela spelet mot dig själv.
  * Fråga 1: Verkar spelet enkelt att vinna?
    * Ja
  * Fråga 2: Om du spelar perfekt som båda spelarna, tror du att någon spelare någonsin vinner?
    * Nej
  * Fråga 3: Tror du att platsen för det första draget påverkar om ett spel vinns eller förloras, om båda spelarna spelar perfekt?
    * Ja

In [ ]:
BOARD_SIZE = 3
board = [[0] * BOARD_SIZE for _ in range(BOARD_SIZE)]

player1_dropdown.options = [
      ('Människa', 'human')
  ]
player2_dropdown.options = [
      ('Människa', 'human')
  ]

# Starta spelet
new_game()

# Tre i rad - Lab 1.1 - Slumpmässig agent

* **Uppgift 1.1.1:** Läs koden.
  * Fråga 1: Vad gör *random.choice(available_moves)*?
  * random.choice() slumpar ut ett element ur listan
* **Uppgift 1.1.2:** Kör kodcellerna och spela mot den slumpmässiga agenten.
  * Fråga 2: Är det enkelt för dig att vinna mot den?
  * Ja
  * Fråga 3: Påverkas svårigheten av vem som får göra det första draget?
  * Nej
* **Uppgift 1.1.3:** Låt den slumpmässiga agenten spela mot sig själv.
  * Fråga 4: Verkar spelet intelligent?
  * Fråga 5: Hur skulle en mer intelligent agent bete sig?
  * Fråga 6: Vilka enkla förbättringar (konceptuellt) skulle göra agenten bättre?
  
(Om AI-agenten ska göra det första draget, tryck på **Steg**.)

In [ ]:
def find_best_move_random(board):
    '''
    Exempelstrategi för en slumpmässig strategi.
    Aktuell spelare används inte (ett tomt indata).
    '''

    # Hämta listan med alla tomma positioner på spelplanen
    available_moves = [(i, j) for i in range(BOARD_SIZE) for j in range(BOARD_SIZE) if board[i][j] == 0]

    if available_moves:
        return random.choice(available_moves)

In [ ]:
# Inställning av spelplanen
BOARD_SIZE = 3
board = [[0] * BOARD_SIZE for _ in range(BOARD_SIZE)]

players = [
    ('Människa', 'human'),
    ('Slumpmässig', 'random')
]

# Rullgardinsmeny för spelare 1 och spelare 2
player1_dropdown.options = players
player2_dropdown.options = players
player2_dropdown.value = players[-1][1]

# Starta spelet
new_game()

# Tre i rad - Lab 1.2 - Din egen agent

**Kommer du kunna slå din egen agent?**

* **Uppgift 1.2.1:** Läs koden.
  * Fråga 1: Vad gör *Regel 1* och hur fungerar den?
* **Uppgift 1.2.2:** Utöka koden för att göra din egen AI-agent.<br/>
  (Minimumkrav: Om spelplanens storlek är 3 och planen är tom, gör det första draget i mitten.)
  * Fråga 1: Vilka är några (1–3) bra regler?
  * Fråga 2: Vilka regler implementerade du?
* **Uppgift 1.2.3:** Kör kodcellerna och spela mot din egen agent.
  * Fråga 2: Är det enkelt för dig att vinna mot den?
  * Fråga 3: Påverkas svårigheten av vem som gör det första draget?
* **Uppgift 1.2.4:** Låt din egen agent spela mot sig själv.
  * Fråga 4: Verkar spelet intelligent?
  * Fråga 5: Hur skulle en mer intelligent agent bete sig?
  * Fråga 6: Vilka enkla förbättringar (konceptuellt) skulle göra agenten bättre?
  
  

In [ ]:
def find_best_move(board, current_player):
    '''
    Funktion som eleverna ska ändra.

    Målet är att implementera regler för hur AI:n ska spela.
    Du har redan fått den första och sista regeln.

    Returnera en tupel (i, j) som blir AI:ns drag. (0, 0) är överst till vänster.
    '''

    # Regel 1: Vinn om möjligt
    for i in range(BOARD_SIZE):
        for j in range(BOARD_SIZE):
            if board[i][j] == 0:
                if check_tile_status(board, i, j, current_player) == current_player:
                    return (i, j)  # Vinnande drag hittades

    # TODO: Lägg till regler
    # ...............

    # Sista regeln: vilket drag som helst
    return find_best_move_random(board)

In [ ]:
# Inställning av spelplanen
BOARD_SIZE = 3
board = [[0] * BOARD_SIZE for _ in range(BOARD_SIZE)]

players = [
    ('Människa', 'human'),
    ('Slumpmässig', 'random'),
    ('Min AI-strategi', 'my_ai_strategy')
]

# Rullgardinsmeny för spelare 1 och spelare 2
player1_dropdown.options = players
player2_dropdown.options = players
player2_dropdown.value = players[-1][1]

# Starta spelet
new_game()

# Tre i rad - Lab 1.3 - Regelbaserad agent

* **Uppgift 1.3.1:** Läs koden.
  * Fråga 1: Förklara strategin hos den regelbaserade agenten med egna ord.
  * Fråga 2: Vad är skillnaden i *tänka framåt* mellan *Regel 1* och *Regel 2*?
  * Fråga 3: Om du måste ta bort två regler, vilka tror du påverkar spelet minst negativt?
* **Uppgift 1.3.2:** Kör kodcellerna och spela mot den regelbaserade agenten.
  * Fråga 4: Är det enkelt för dig att vinna mot den?
  * Fråga 5: Påverkas svårigheten av vem som gör det första draget?
* **Uppgift 1.3.3:** Låt den regelbaserade agenten spela mot den slumpmässiga agenten.
  * Fråga 6: Vilken agent verkar mer intelligent?
  * Fråga 7: Spelar det någon roll vilken agent som börjar?
* **Uppgift 1.3.4:** Låt den regelbaserade agenten spela mot sig själv.
  * Fråga 8: Verkar spelet intelligent?
  * Fråga 9: Hur skulle en mer intelligent agent bete sig?
  * Fråga 10: Vilka enkla förbättringar (konceptuellt) skulle göra agenten bättre?


In [ ]:
def find_best_move_rule_based(board, current_player):
    '''
    Exempelfunktion för en regelbaserad strategi, använder check_tile_status.
    '''
    opponent = 2 if current_player == 1 else 1

    # Regel 1: Vinn om möjligt
    for i in range(BOARD_SIZE):
        for j in range(BOARD_SIZE):
            if board[i][j] == 0:
                if check_tile_status(board, i, j, current_player) == current_player:
                    return (i, j)  # Vinnande drag hittades

    # Regel 2: Blockera motståndarens vinst
    for i in range(BOARD_SIZE):
        for j in range(BOARD_SIZE):
            if board[i][j] == 0:
                if check_tile_status(board, i, j, opponent) == opponent:
                    return (i, j)  # Blockerande drag hittades

    # Regel 3: Ta mitten om den är ledig (bara för 3x3 spelplan just nu)
    if BOARD_SIZE == 3 and board[1][1] == 0:
        return (1, 1)

    # Regel 4: Ta ett hörn om något är ledigt
    for i, j in [(0, 0), (0, BOARD_SIZE - 1), (BOARD_SIZE - 1, 0), (BOARD_SIZE - 1, BOARD_SIZE - 1)]:
        if board[i][j] == 0:
            return (i, j)

    # Regel 5: Ta valfri ledig ruta
    return find_best_move_random(board)

In [ ]:
# Inställning av spelplanen
BOARD_SIZE = 3
board = [[0] * BOARD_SIZE for _ in range(BOARD_SIZE)]

players = [
    ('Människa', 'human'),
    ('Slumpmässig', 'random'),
    ('Min AI-strategi', 'my_ai_strategy'),
    ('Regelbaserad', 'rule_based')
]

# Rullgardinsmeny för spelare 1 och spelare 2
player1_dropdown.options = players
player2_dropdown.options = players
player2_dropdown.value = players[-1][1]

# Starta spelet
new_game()

# Tre i rad - Lab 1.4 - Planeringsagent (sökning)

* **Uppgift 1.4.1:** Läs koden.
  * Fråga 1: Förklara strategierna hos de sökbaserade agenterna med egna ord.

* **Uppgift 1.4.2:** Kör kodcellerna och spela mot den sökbaserade agenten.
  * Fråga 2: Är det enkelt för dig att vinna mot dem?
  * Fråga 3: Varför passar inte sökbaserade agenter som bara letar efter vinnande drag för spel mot motståndare?
  * Fråga 4: Hur tror du att en sökbaserad strategi kan förbättras för att spela bättre?



In [ ]:

from collections import deque

def find_best_move_dfs(board, player, max_depth=6):
    '''
    Hittar det bästa draget för AI:n med Djupet-Först-sökning (DFS).
    Den här funktionen söker rekursivt i spelträdet upp till ett visst djup
    och väljer det första draget som leder till vinst.
    Om ingen vinnande väg hittas inom djupgränsen väljs ett slumpmässigt drag.
    '''
    opponent = 2 if player == 1 else 1

    def dfs(board, current_player, depth):
        winner = check_winner(board)
        if winner == player:
            return True  # Vinnande väg hittad
        elif winner == opponent or winner == 3:
            return False  # Motståndaren vinner eller oavgjort
        elif depth == max_depth:
            return False  # Maxdjup nått utan att hitta vinst

        for i in range(BOARD_SIZE):
            for j in range(BOARD_SIZE):
                if board[i][j] == 0:
                    # Gör ett drag
                    board[i][j] = current_player
                    found_win = dfs(board, 2 if current_player == 1 else 1, depth + 1)
                    # Ångra draget
                    board[i][j] = 0

                    if current_player == player and found_win:
                        return True  # Vinnande väg hittad
        return False  # Ingen vinnande väg hittad i den här grenen

    # Prova alla möjliga drag
    for i in range(BOARD_SIZE):
        for j in range(BOARD_SIZE):
            if board[i][j] == 0:
                # Gör ett drag
                board[i][j] = player
                found_win = dfs(board, opponent, 1)
                # Ångra draget
                board[i][j] = 0

                if found_win:
                    return (i, j)  # Returnera det första draget som leder till vinst

    # Ingen vinnande väg hittad, välj ett slumpmässigt drag
    return find_best_move_random(board)

def find_best_move_bfs(board, current_player, max_depth = 5):
    '''
    Hittar det bästa draget för AI:n med Bredden-Först-sökning (BFS).
    Den här funktionen utforskar spelträdet nivå för nivå för att hitta det snabbaste vinnande draget.
    '''
    opponent = 2 if current_player == 1 else 1
    queue = deque()

    # Initiera kön med alla möjliga drag för den aktuella spelaren
    for i in range(BOARD_SIZE):
        for j in range(BOARD_SIZE):
            if board[i][j] == 0:
                new_board = [row[:] for row in board]
                new_board[i][j] = current_player
                queue.append((new_board, (i, j), current_player, 1))  # (spelplansstatus, drag, spelare, djup)

    best_move = None
    min_depth = float('inf')

    while queue:
        current_board, move, player, depth = queue.popleft()
        winner = check_winner(current_board)

        if winner == current_player:
            # Vinnande drag hittades
            if depth < min_depth:
                min_depth = depth
                best_move = move
            continue
        elif winner == opponent or winner == 3:
            continue # Hoppa över oavgjort eller motståndarens vinst

        if depth >= max_depth:
            continue # Stoppa vid maxdjup

        # Generera möjliga drag för nästa spelare
        next_player = opponent if player == current_player else current_player

        for i in range(BOARD_SIZE):
            for j in range(BOARD_SIZE):
                if current_board[i][j] == 0:
                    new_board = [row[:] for row in current_board]
                    new_board[i][j] = next_player
                    queue.append((new_board, move, next_player, depth + 1))

    if best_move is not None:
        return best_move
    else:
        # Om inget vinnande drag hittas, välj ett slumpmässigt drag
        return find_best_move_random(board)



In [ ]:
# Inställning av spelplanen
BOARD_SIZE = 3
board = [[0] * BOARD_SIZE for _ in range(BOARD_SIZE)]

players = [
    ('Människa', 'human'),
    ('Slumpmässig', 'random'),
    ('Min AI-strategi', 'my_ai_strategy'),
    ('Regelbaserad', 'rule_based'),
    ('Djupet-Först-sökning', 'dfs'),
    ('Bredden-Först-sökning', 'bfs')
]

# Rullgardinsmeny för spelare 1 och spelare 2
player1_dropdown.options = players
player2_dropdown.options = players
player2_dropdown.value = 'dfs'

# Starta spelet
new_game()

# Tre i rad - Lab 1.5 - Motståndarspelarens planeringsagent (Minimax)

* **Uppgift 1.5.1:** Läs koden.
  * Fråga 1: Vilka är de möjliga returvärdena från funktionen *minimax(...)* och vad betyder de?
  * Fråga 2: Hur många gånger anropas funktionen *minimax(...)* av funktionen *find_best_move_min_max(...)*?
  * Fråga 3: Tror du att *minimax(...)* anropas fler gånger från sig själv än från *find_best_move_min_max(...)*?
  * Fråga 4: Tror du att den här agenten är snabbare, långsammare eller liknande i beräkningstid?
* **Uppgift 1.5.2:** Kör kodcellerna och spela mot motståndarspelarens planeringsagent.
  * Fråga 5: Är det enkelt för dig att vinna mot den?
  * Fråga 6: Påverkas svårighetern av vem som gör det första draget?
* **Uppgift 1.5.3:** Låt motståndarspelarens planeringsagent (Minimax) spela mot de andra agenterna.
  * Fråga 7: Vilken agent verkar mer intelligent?
  * Fråga 8: Spelar det någon roll vilken agent som börjar?
  * Fråga 9: Verkar det spela roll om du ökar spelplanens storlek till 4x4?
  * Fråga 10: Hur påverkar parametern max_depth minimax-algoritmen?

* **Uppgift 1.5.4:** Låt agenten spela mot sig själv.
  * Fråga 10: Verkar spelet intelligent?

In [ ]:
def evaluate_board(board, current_player):
    opponent = 2 if current_player == 1 else 1
    score = 0

    center = (BOARD_SIZE - 1) // 2

    # Positionspoäng
    for i in range(BOARD_SIZE):
        for j in range(BOARD_SIZE):
            if board[i][j] == current_player:
                # Mittposition
                if i == center and j == center:
                    score += 3  # Ge högt värde till mitten
                # Hörnpositioner
                elif (i == 0 or i == BOARD_SIZE - 1) and (j == 0 or j == BOARD_SIZE - 1):
                    score += 2  # Ge medelhögt värde till hörnen
                else:
                    score += 1  # Ge lägre värde till kanterna
            elif board[i][j] == opponent:
                # Dra av positionspoäng för motståndarens brickor
                if i == center and j == center:
                    score -= 3
                elif (i == 0 or i == BOARD_SIZE - 1) and (j == 0 or j == BOARD_SIZE - 1):
                    score -= 2
                else:
                    score -= 1

    # Utvärdera möjliga vinnande linjer för båda spelarna
    lines = []

    # Rader och kolumner
    for i in range(BOARD_SIZE):
        lines.append(board[i])    # Rad
        lines.append([board[j][i] for j in range(BOARD_SIZE)])    # Kolumn

    # Diagonaler
    lines.append([board[k][k] for k in range(BOARD_SIZE)])
    lines.append([board[k][BOARD_SIZE-1-k] for k in range(BOARD_SIZE)])

    for line in lines:
        score += evaluate_line(line, current_player, opponent)

    return score


def evaluate_line(line, player, opponent):
    score = 0
    player_count = sum(1 for x in line if x == player)
    opponent_count = sum(1 for x in line if x == opponent)
    empty_count = sum(1 for x in line if x == 0)

    if player_count == 2 and empty_count == 1:
        score += 10  # Möjlig vinnande linje för AI
    elif player_count == 1 and empty_count == 2:
        score += 1   # Svag möjlig linje för AI
    elif opponent_count == 2 and empty_count == 1:
        score -= 9   # Möjlig vinnande linje för motståndaren (blockera den)
    elif opponent_count == 1 and empty_count == 2:
        score -= 1   # Svag möjlig linje för motståndaren

    return score

def minimax(board, depth, is_maximizing, max_depth, current_player, alpha=float('-inf'), beta=float('inf'), use_alpha_beta=False):
    opponent = 2 if current_player == 1 else 1

    winner = check_winner(board)
    if winner == current_player:
        return 100 - depth  # Föredra snabba vinster
    elif winner == opponent:
        return -100 + depth  # Föredra långa förluster
    elif winner == 3:
        return 0  # Oavgjort
    elif depth == max_depth:
        # Använd utvärderingsfunktionen
        return evaluate_board(board, current_player)

    if is_maximizing:
        best_score = float('-inf')
        for i in range(BOARD_SIZE):
            for j in range(BOARD_SIZE):
                if board[i][j] == 0:
                    board[i][j] = current_player
                    score = minimax(board, depth + 1, False, max_depth, current_player, alpha, beta, use_alpha_beta)
                    board[i][j] = 0
                    best_score = max(score, best_score)
                    if use_alpha_beta:
                        alpha = max(alpha, best_score)
                        if beta <= alpha:
                            break
            if use_alpha_beta and beta <= alpha:
                break
        return best_score
    else:
        best_score = float('inf')
        for i in range(BOARD_SIZE):
            for j in range(BOARD_SIZE):
                if board[i][j] == 0:
                    board[i][j] = opponent
                    score = minimax(board, depth + 1, True, max_depth, current_player, alpha, beta, use_alpha_beta)
                    board[i][j] = 0
                    best_score = min(score, best_score)
                    if use_alpha_beta:
                        beta = min(beta, best_score)
                        if beta <= alpha:
                            break
            if use_alpha_beta and beta <= alpha:
                break
        return best_score

def find_best_move_min_max(board, current_player, use_alpha_beta=True, max_depth=8):
    '''
    Hitta det bästa draget med Minimax-algoritmen och valfri alfa-beta-beskärning.
    AI:n utvärderar drag genom att maximera sin egen potential och minimera motståndarens.
    '''
    best_move = None
    best_score = float('-inf')

    if BOARD_SIZE >= 4:
      max_depth = 5

    opponent = 2 if current_player == 1 else 1

    # Initiera alfa- och beta-värden för alfa-beta-beskärning
    alpha = float('-inf')
    beta = float('inf')

    for i in range(BOARD_SIZE):
        for j in range(BOARD_SIZE):
            if board[i][j] == 0:
                # Gör draget
                board[i][j] = current_player
                # Anropa minimax rekursivt och välj det högsta värdet
                score = minimax(board, 0, False, max_depth, current_player, alpha, beta, use_alpha_beta)
                # Ångra draget
                board[i][j] = 0

                if score > best_score:
                    best_score = score
                    best_move = (i, j)

                # Uppdatera alfa-värdet om alfa-beta-beskärning är aktiverad
                if use_alpha_beta:
                    alpha = max(alpha, best_score)
                    if beta <= alpha:
                        break  # Alfa-avskärning
        if use_alpha_beta and beta <= alpha:
            break  # Alfa-avskärning

    return best_move

In [ ]:
# Inställning av spelplanen
BOARD_SIZE = 3
board = [[0] * BOARD_SIZE for _ in range(BOARD_SIZE)]

players = [
    ('Människa', 'human'),
    ('Slumpmässig', 'random'),
    ('Min AI-strategi', 'my_ai_strategy'),
    ('Regelbaserad', 'rule_based'),
    ('Djupet-Först-sökning', 'dfs'),
    ('Bredden-Först-sökning', 'bfs'),
    ('MinMax', 'minimax')
]

# Rullgardinsmeny för spelare 1 och spelare 2
player1_dropdown.options = players
player2_dropdown.options = players
player2_dropdown.value = 'minimax'

# Starta spelet
new_game()

# Tre i rad - Lab 1.6 - Stokastisk planeringsagent (MCTS)

**(Frivillig)**

* **Uppgift 1.6.1:** Implementera MCTS
  * Fråga 1: Vilken källa använde du?
  * Fråga 2: Vilka skillnader mot Minimax förväntar du dig?
* **Uppgift 1.6.2:** Kör kodcellerna och spela mot den stokastiska planeringsagenten (MCTS).
  * Fråga 5: Är det enkelt för dig att vinna mot den?
  * Fråga 6: Påverkas svårigheten av vem som gör det första draget?
* **Uppgift 1.5.3:** Låt den stokastiska planeringsagenten (MCTS) spela mot de andra agenterna.
  * Fråga 7: Vilken agent verkar mer intelligent?
  * Fråga 8: Spelar det någon roll vilken agent som börjar?
  * Fråga 9: Spelar det någon roll om du ökar spelplanens storlek till 4x4?
* **Uppgift 1.5.4:** Låt den stokastiska planeringsagenten (MCTS) spela mot sig själv.
  * Fråga 10: Verkar spelet intelligent?